# Execution on a quantum processing unit (QPU)

- `qiskit` base package with most of the language features (statevectors, operators, circuits, ...) and visualization (`qiskit[visualization]`).

- `qiskit-ibm-runtime` to interact with cloud resources (and therefore with real quantum processors).

- `qiskit-aer` to efficiently simulate code on your own machine. (It provides different simulation methods and can accelerate fake backends locally: see https://quantum.cloud.ibm.com/docs/en/guides/local-testing-mode)

Checking installed versions:

In [ ]:
from qiskit import __version__
print (__version__)

from qiskit_ibm_runtime import __version__
print (__version__)

from qiskit_aer import __version__
print (__version__)

## 1. Creating a circuit

In [ ]:
from qiskit.circuit import QuantumCircuit

circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0,1)
circuit.measure_all()

circuit.draw(output="mpl", filename="circuit.pdf")

## 2. Choosing the execution platform

Several options available:
1. Real quantum processor
2. Fake quantum processor (simulator that shares similar constraints: noise and limited number of qubits)
3. AER simulator

Execution can be done in the cloud (for example on real QPUs) or locally (AER simulator). For fake QPUs, both options exist. **show**

In [ ]:
backend_option = "fake"

match backend_option:
    case "qpu": 
        print("Selecting QPU backend")

        from qiskit_ibm_runtime import QiskitRuntimeService
        service = QiskitRuntimeService(channel="ibm_quantum_platform", token="rE-h6mLz-xAeWSRuUY2nx28xVNxnquE-qjN0gFOhXVGi")
        backend = service.least_busy(simulator=False, operational=True)

    case "fake":
        print("Selecting fake backend")

        from qiskit_ibm_runtime.fake_provider import FakeManilaV2
        backend = FakeManilaV2()

    case "aer":
        print("Selecting AER backend")

        from qiskit_aer import AerSimulator
        backend = AerSimulator()
        
    case _:
        print("Unknown option")

In [ ]:
# from qiskit_ibm_runtime import QiskitRuntimeService
# service = QiskitRuntimeService(channel="ibm_quantum_platform", token="rE-h6mLz-xAeWSRuUY2nx28xVNxnquE-qjN0gFOhXVGi")

# backend = service.least_busy(simulator=False, operational=True)

In [ ]:
# from qiskit_ibm_runtime.fake_provider import FakeManilaV2

# backend = FakeManilaV2()

In [ ]:
# from qiskit_aer import AerSimulator

# backend = AerSimulator()

## 3. "Compilation"

The next step is to adapt and optimize the circuit for the chosen platform. Two interfaces:

1. `qiskit.compiler` offers a simple interface, similar to a classical compiler
2. `qiskit.transpiler` allows customizing the transpilation steps

In [ ]:
from qiskit.compiler import transpile
circuit_transpiled = transpile(circuit, backend, optimization_level=3)

#from qiskit.transpiler import generate_preset_pass_manager
#pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
#circuit_transpiled = pm.run(circuit)

print("Transpiled for backend =", backend.name)
circuit_transpiled.draw(output="mpl", filename="circuit_transpiled.pdf")

## 4. Execution

**TODO:** Describe the procedure when interacting with real QPUs. In general, show how to retrieve jobs asynchronously when using cloud resources.


In [ ]:
from qiskit_ibm_runtime import SamplerV2 as Sampler
sampler = Sampler(backend)
job = sampler.run([circuit_transpiled], shots=1024)

job_id = job.job_id()
print("Job id: ", job_id)

## 5. Analysis of results

In [ ]:
#job = service.job(job_id)
result = job.result()

from qiskit.visualization import plot_histogram
plot_histogram(result[0].data.meas.get_counts())

- With the AER simulator, the pairs $\ket{01}$ and $\ket{10}$ will never be measured.
- Current QPUs are susceptible to a high error rate: the quantum state is progressively spread across all states of the computational basis, leading to unexpected results during measurement.